# 05. Runnable 기본 조합 (LCEL)

- LangChain에서의 Runnable은 입력값을 받아 특정 작업을 수행하고 출력값을 반환하는 실행 가능한 구성요소임.
- 이를 활용하면 프롬프트, 모델, 출력 파서, 사용자 정의 함수, 병렬 처리 흐름을 직관적인 파이프라인으로 표현할 수 있음.
- LCEL(LangChain Expression Language)은 `|` 연산자로 컴포넌트를 체인처럼 연결함.
- Runnable 조합을 사람이 읽기 쉬운 파이프라인 문법으로 표현한 것.


```
prompt | llm | parser
```

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, model="gpt-4.1", temperature=0)
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


## 1. 기본 체인: prompt | llm | parser

In [2]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}에 대해 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# | 연산자로 체인 구성
chain = prompt | llm | parser

result = chain.invoke({"topic": "벡터 데이터베이스"})
print(result)

벡터 데이터베이스는 텍스트, 이미지 등 다양한 데이터를 고차원 벡터로 변환하여 유사도 기반 검색과 분석을 효율적으로 수행하는 데이터베이스입니다.


## 2. 체인 스트리밍
- LangChain 체인 실행 결과를 스트리밍 방식으로 조금씩 받아 출력하는 코드

In [3]:
for chunk in chain.stream({"topic": "임베딩"}):
    # 현재 받은 응답 조각을 출력 후 줄바꿈 없이, 버퍼에 쌓아두지 않고 즉시 출력
    print(chunk, end="", flush=True)

임베딩이란 단어나 문장, 이미지 등의 데이터를 고정된 크기의 연속적인 수치 벡터로 변환하여 컴퓨터가 이해하고 처리할 수 있도록 하는 기법입니다.

In [4]:
for chunk in chain.stream({"topic": "임베딩"}):
    # Python이 출력 내용을 잠시 버퍼에 모아두었다가 나중에 보여줌
    print(chunk, end="", flush=False)

임베딩이란 단어나 문장, 이미지 등의 데이터를 고정된 크기의 연속적인 수치 벡터로 변환하여 컴퓨터가 이해하고 처리할 수 있도록 하는 방법입니다.

# 3. Runnable의 공통 실행 메서드
- 입력 값을 받아 어떤 작업을 수행하고, 출력값을 반환하는 표준 실행 단위
- 프롬프트, LLM, 출력파서, Retriever, 사용자 정의 함수 등을 하나의 실행 규칙으로 연결하기 위한 표준 인터페이스
- 기본 : 입력 → Runnable → 출력
- 여러 Runnable 연결 : 입력 → prompt → llm → parser → custom function → 최종 출력 

- Runnable의 공통 실행 메서드 

| 메서드                    | 의미              | 사용 상황           |
| ---------------------- | --------------- | --------------- |
| `invoke()`             | 단일 입력 실행        | 가장 기본 실행        |
| `ainvoke()`            | 비동기 단일 실행       | async 환경        |
| `batch()`              | 여러 입력을 한 번에 실행  | 여러 질문 일괄 처리     |
| `abatch()`             | 비동기 batch 실행    | 대량 처리           |
| `stream()`             | 결과를 조각 단위로 스트리밍 | 챗봇 실시간 출력       |
| `astream()`            | 비동기 스트리밍        | 웹소켓, SSE 등      |
| `batch_as_completed()` | 완료되는 순서대로 결과 반환 | 입력별 처리 시간이 다를 때 |


## 3. RunnablePassthrough - 입력을 그대로 전달
- RunnablePassthrough는 입력값을 거의 그대로 통과시킴

In [5]:
from langchain_core.runnables import RunnablePassthrough

# 입력값을 그대로 흘려보내기
chain = RunnablePassthrough()
result = chain.invoke({"topic": "RAG"})
print(result)

{'topic': 'RAG'}


- `.assign()`은 기존 딕셔너리에 새로운 필드를 추가할 때 사용함

In [6]:
# 실용 예: 입력을 그대로 유지하면서 추가 키 삽입
from langchain_core.runnables import RunnablePassthrough

upper=lambda x: x["word"].upper()
chain = RunnablePassthrough.assign(upper=upper)

result = chain.invoke({"word": "hello"})
print(result)  # {'word': 'hello', 'upper': 'HELLO'}

{'word': 'hello', 'upper': 'HELLO'}


## 4. RunnableLambda - 함수를 Runnable로 감싸기
- RunnableLambda는 일반 파이썬 함수를 Runnable로 감싸는 객체
- RunnableLambda는 체인 중간이나 마지막에 후처리 로직을 넣을 때 주로 사용

In [7]:
from langchain_core.runnables import RunnableLambda

# 문자열 -> 문자열 편집 반환 함수
def add_prefix(text: str) -> str:
    return f"[결과] {text}"

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}이 뭔가요?")
])

chain = prompt | llm | StrOutputParser() | RunnableLambda(add_prefix)

result = chain.invoke({"topic": "LangGraph"})
print(result)

[결과] LangGraph는 LLM(대형 언어 모델) 기반 애플리케이션을 구축하기 위한 오픈소스 프레임워크입니다. LangChain 팀에서 개발했으며, 기존의 LangChain이 체인(Chain)이나 에이전트(Agent) 중심의 워크플로우를 제공했다면, LangGraph는 **그래프(Graph)** 구조를 활용해 더 복잡하고 유연한 대화 흐름이나 워크플로우를 설계할 수 있도록 도와줍니다.

### 주요 특징

- **상태 기반(Stateful) 워크플로우**: LangGraph는 각 노드가 상태를 받아 처리하고, 다음 노드로 상태를 넘기는 방식으로 동작합니다. 이를 통해 대화의 맥락이나 작업의 중간 결과를 쉽게 관리할 수 있습니다.
- **분기와 반복(Branching & Looping)**: 그래프 구조이기 때문에, 조건에 따라 분기하거나, 반복적인 처리를 쉽게 구현할 수 있습니다.
- **멀티에이전트 시스템**: 여러 LLM 에이전트가 서로 상호작용하는 복잡한 시스템을 설계할 수 있습니다.
- **LangChain과의 통합**: LangChain에서 제공하는 다양한 도구, 체인, 에이전트와 쉽게 연동할 수 있습니다.

### 예시

예를 들어, LangGraph를 사용하면 다음과 같은 워크플로우를 만들 수 있습니다.

1. 사용자의 질문을 받아서
2. 질문의 유형을 분류하고
3. 필요한 경우 외부 API를 호출하거나, 여러 에이전트가 협업해서 답변을 생성하고
4. 결과를 종합해서 사용자에게 전달

이런 복잡한 흐름을 그래프 형태로 설계할 수 있습니다.

### 공식 문서

- [LangGraph 공식 문서](https://langgraph.readthedocs.io/)
- [LangChain 블로그의 LangGraph 소개](https://blog.langchain.dev/introducing-langgraph/)

---

**요약:**  
LangGraph는 LLM 기반 애플리케이션에서 복잡한 대화 흐름이나 멀티에이전트 시스템을 그래프 구조로 쉽게 설계

## 5. 병렬 실행 - RunnableParallel

딕셔너리 형태로 여러 체인을 동시에 실행합니다.

In [8]:
from langchain_core.runnables import RunnableParallel

prompt_ko = ChatPromptTemplate.from_messages([("human", "{topic}을 한국어로 설명해줘")])
prompt_en = ChatPromptTemplate.from_messages([("human", "Explain {topic} in English")])

chain_ko = prompt_ko | llm | StrOutputParser()
chain_en = prompt_en | llm | StrOutputParser()

# 두 체인을 병렬로 실행하여 각각 한국어와 영어 설명 생성
parallel_chain = RunnableParallel(
    korean=chain_ko,
    english=chain_en
)

result = parallel_chain.invoke({"topic": "RAG"})
print("[한국어]", result["korean"])
print("[English]", result["english"])

[한국어] 물론입니다!  
RAG는 "Retrieval-Augmented Generation"의 약자로, 한국어로는 **"검색 기반 생성"** 또는 **"검색 보강 생성"**이라고 번역할 수 있습니다.

### RAG의 개념
RAG는 인공지능 언어 모델이 답변을 생성할 때, 단순히 학습된 지식만 사용하는 것이 아니라, **외부 데이터베이스나 문서에서 관련 정보를 검색**해서 그 정보를 바탕으로 더 정확하고 신뢰성 있는 답변을 생성하는 기술입니다.

### RAG의 동작 방식
1. **검색 단계(Retrieval):**  
   사용자의 질문이 들어오면, 먼저 외부 문서(예: 위키피디아, 사내 문서, 논문 등)에서 관련된 정보를 검색합니다.

2. **생성 단계(Generation):**  
   검색된 정보를 바탕으로, 언어 모델이 답변을 생성합니다.

이렇게 하면, 모델이 학습 데이터에 없는 최신 정보나 구체적인 내용을 포함한 답변을 할 수 있습니다.

### RAG의 장점
- **최신 정보 반영:** 모델이 학습 이후에 생긴 정보도 검색을 통해 활용할 수 있습니다.
- **정확성 향상:** 검색된 신뢰할 수 있는 자료를 바탕으로 답변하므로, 더 정확한 답변이 가능합니다.
- **메모리 효율:** 모든 정보를 모델에 학습시키지 않아도 되므로, 모델의 크기를 줄일 수 있습니다.

### 예시
예를 들어, "2024년 올림픽 개최지는 어디인가요?"라는 질문이 들어오면,  
RAG 모델은 먼저 관련 문서를 검색해서 "2024년 올림픽은 파리에서 개최된다"는 정보를 찾고,  
그 정보를 바탕으로 답변을 생성합니다.

---

**요약:**  
RAG는 "검색 기반 생성" 기술로, 외부에서 정보를 검색해와서 그 정보를 바탕으로 답변을 만들어내는 인공지능 기술입니다.  
이 방식은 최신 정보 반영과 정확성 향상에 큰 장점이 있습니다.
[English] Certainly! **RAG** stands for **Retrieval-Augmented Generation**.


## 실습 문제
Q2. RunnableLambda를 활용해 LLM 응답을 받아 단어 수를 세는 함수를 파이프라인 끝에 추가해보세요. 결과로 {'text': '...', 'word_count': 42} 형태로 반환하세요.

In [9]:
from langchain_core.runnables import RunnableLambda

# 1. 프롬프트 구성
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 초보자도 이해할 수 있게 설명해줘."
)

# 2. LLM 생성
llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, 
    model="gpt-4.1",
    temperature=0
)

# 3. 출력 파서 생성
parser = StrOutputParser()

# 4. LLM 응답을 받아 단어 수를 세는 함수
def count_words(text: str) -> dict:
    return {
        "text": text,
        "word_count": len(text.split())
    }

# 5. RunnableLambda로 함수 감싸기
word_count_runnable = RunnableLambda(count_words)

# 6. LCEL 체인 구성
chain = prompt | llm | parser | word_count_runnable

# 7. 실행
result = chain.invoke({"topic": "LangChain Runnable"})

print(result["word_count"])  # 설명의 단어 수
print("-"*20)
print(result["text"])        # LLM이 생성한 설명

245
--------------------
네! LangChain의 **Runnable**에 대해 초보자도 이해할 수 있게 쉽게 설명해드릴게요.

---

## 1. LangChain이란?
LangChain은 **챗봇**이나 **AI 앱**을 만들 때, 여러 AI 모델(예: ChatGPT), 데이터베이스, 도구 등을 쉽게 연결해서 쓸 수 있게 도와주는 파이썬/자바스크립트 라이브러리예요.

---

## 2. Runnable이란?
**Runnable**은 LangChain에서 "실행할 수 있는 것"을 뜻해요.  
쉽게 말해, **입력을 받아서 출력을 내보내는 함수나 객체**라고 생각하면 돼요.

### 예시로 설명해볼게요!

#### 1) 함수와 비슷해요
- 예를 들어, `f(x) = x + 1` 이라는 함수가 있으면,
- 입력으로 3을 넣으면, 출력은 4가 되죠.

#### 2) LangChain의 Runnable
- LangChain에서는 이런 "입력 → 처리 → 출력" 과정을 **Runnable**이라는 이름으로 부릅니다.
- 예를 들어, "질문을 받아서 답변을 생성하는 AI"도 Runnable이에요.

---

## 3. Runnable의 장점
- **조립 가능**: 여러 Runnable을 연결해서 복잡한 작업을 만들 수 있어요. (예: 질문 → 검색 → 요약 → 답변)
- **재사용 가능**: 한 번 만든 Runnable을 여러 곳에서 쓸 수 있어요.
- **동일한 인터페이스**: 다양한 작업(예: LLM 호출, 데이터베이스 검색 등)을 같은 방식으로 다룰 수 있어요.

---

## 4. 실제 코드 예시 (파이썬)

```python
from langchain.llms import OpenAI
from langchain.schema.runnable import RunnableLambda

# 1. LLM(예: ChatGPT)도 Runnable입니다.
llm = OpenAI(model="gpt-3.5-turbo")

# 2. 직접 만든 함수도 Runn